In [ ]:
from pymilvus import MilvusClient
import math
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer


Data Processing

In [ ]:
loader = PyPDFLoader('sciencerag.pdf')

document = loader.load()

document

[Document(page_content='Photosynthesis in Plants\nPhotosynthesis is a biological process that occurs in plants, algae, and some bacteria, where light\nenergy is \nconverted into chemical energy in the form of glucose (a sugar) and oxygen. This process is vital for\nthe survival \nof life on Earth, as it provides the base for the food chain and releases oxygen into the atmosphere.\n### The Process of Photosynthesis\nPhotosynthesis takes place primarily in the leaves of plants, within specialized organelles called\nchloroplasts.\nThe process can be broken down into two main stages: the light-dependent reactions and the Calvin\ncycle \n(light-independent reactions).\n1. **Light-dependent reactions (Occurs in thylakoid membranes of chloroplasts)**\nThese reactions occur when light is absorbed by chlorophyll pigments in the thylakoid membranes.\nWater molecules are \nsplit into oxygen, protons (H+), and electrons through photolysis. The electrons move through the\nelectron \ntransport chain

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

chunks = text_splitter.split_documents(document)

for chunk in chunks:
    print(chunk)
    print("-------------")

page_content='Photosynthesis in Plants\nPhotosynthesis is a biological process that occurs in plants, algae, and some bacteria, where light\nenergy is \nconverted into chemical energy in the form of glucose (a sugar) and oxygen. This process is vital for\nthe survival \nof life on Earth, as it provides the base for the food chain and releases oxygen into the atmosphere.\n### The Process of Photosynthesis\nPhotosynthesis takes place primarily in the leaves of plants, within specialized organelles called\nchloroplasts.\nThe process can be broken down into two main stages: the light-dependent reactions and the Calvin\ncycle \n(light-independent reactions).\n1. **Light-dependent reactions (Occurs in thylakoid membranes of chloroplasts)**\nThese reactions occur when light is absorbed by chlorophyll pigments in the thylakoid membranes.\nWater molecules are \nsplit into oxygen, protons (H+), and electrons through photolysis. The electrons move through the\nelectron' metadata={'source': 'scien

In [ ]:
embedding = SentenceTransformer("Alibaba-NLP/gte-large-en-v1.5", trust_remote_code=True)

In [38]:
sources = [chunk.metadata['source'] for chunk in chunks]
pages = [chunk.metadata['page'] for chunk in chunks]
page_contents = [chunk.page_content for chunk in chunks]
content_embedding = [embedding.encode(chunk.page_content) for chunk in chunks]

Milvus Deployment

In [ ]:
client = MilvusClient(uri="http://localhost:19530")


In [ ]:
if client.has_collection(collection_name="demo_collection"):
    client.drop_collection(collection_name="demo_collection")
client.create_collection(
    collection_name="demo_collection",
    dimension=1024,
)


In [51]:
data = [
    {"id": i, "vector": content_embedding[i], "text": page_contents[i], "subject": "science"}
    for i in range(len(content_embedding))
]

In [52]:
print("Data has", len(data), "entities, each with fields: ", data[0].keys())
print("Vector dim:", len(data[0]["vector"]))

Data has 10 entities, each with fields:  dict_keys(['id', 'vector', 'text', 'subject'])
Vector dim: 1024


In [53]:
res = client.insert(collection_name="demo_collection", data=data)

print(res)


{'insert_count': 10, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]}


Cosine Similarity Handler
Source: "https://oneuptime.com/blog/post/2026-01-30-cosine-similarity/view"

In [73]:
def dot_product(vec_a: list[float], vec_b: list[float]) -> float:
    """Calculate the dot product of two vectors."""
    if len(vec_a) != len(vec_b):
        raise ValueError("Vectors must have the same dimension")
    return sum(a * b for a, b in zip(vec_a, vec_b))

def magnitude(vec: list[float]) -> float:
    """Calculate the magnitude (L2 norm) of a vector."""
    return math.sqrt(sum(x * x for x in vec))

def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    """
    Calculate cosine similarity between two vectors.

    Args:
        vec_a: First vector
        vec_b: Second vector

    Returns:
        Cosine similarity value between -1 and 1
    """
    dot = dot_product(vec_a, vec_b)
    mag_a = magnitude(vec_a)
    mag_b = magnitude(vec_b)

    # Handle zero vectors
    if mag_a == 0 or mag_b == 0:
        return 0.0

    return dot / (mag_a * mag_b)

In [71]:
sourcevectors = embedding.encode(chunks[0].page_content)
print(chunks[0].page_content)
print(sourcevectors)

Photosynthesis in Plants
Photosynthesis is a biological process that occurs in plants, algae, and some bacteria, where light
energy is 
converted into chemical energy in the form of glucose (a sugar) and oxygen. This process is vital for
the survival 
of life on Earth, as it provides the base for the food chain and releases oxygen into the atmosphere.
### The Process of Photosynthesis
Photosynthesis takes place primarily in the leaves of plants, within specialized organelles called
chloroplasts.
The process can be broken down into two main stages: the light-dependent reactions and the Calvin
cycle 
(light-independent reactions).
1. **Light-dependent reactions (Occurs in thylakoid membranes of chloroplasts)**
These reactions occur when light is absorbed by chlorophyll pigments in the thylakoid membranes.
Water molecules are 
split into oxygen, protons (H+), and electrons through photolysis. The electrons move through the
electron
[-0.35381526  0.22236973  0.92178565 ...  0.28788584 -1.4

In [60]:
query_vectors = embedding.encode(["What is Photosynthesis?"])
query_vectors

array([[-0.5244754 ,  0.68444866,  0.19398288, ..., -0.15070227,
        -1.5413524 ,  0.44882244]], dtype=float32)

Cosine Similarity Without High Level Library

In [ ]:
similarity = cosine_similarity(sourcevectors, query_vectors[0])
print(f"Cosine Similarity: {similarity:.4f}")

Cosine Similarity: 0.7335


Similarity search using Milvus.
Distance is exactly the same.

In [81]:
query_vectors = embedding.encode(["What is Photosynthesis?"])

res = client.search(
    collection_name="demo_collection",  # target collection
    data=query_vectors,  # query vectors
    limit=10,  # number of returned entities
    output_fields=["text", "subject"],  # specifies fields to be returned
)

print(res)


[[{'id': 0, 'distance': 0.7335335612297058, 'entity': {'text': 'Photosynthesis in Plants\nPhotosynthesis is a biological process that occurs in plants, algae, and some bacteria, where light\nenergy is \nconverted into chemical energy in the form of glucose (a sugar) and oxygen. This process is vital for\nthe survival \nof life on Earth, as it provides the base for the food chain and releases oxygen into the atmosphere.\n### The Process of Photosynthesis\nPhotosynthesis takes place primarily in the leaves of plants, within specialized organelles called\nchloroplasts.\nThe process can be broken down into two main stages: the light-dependent reactions and the Calvin\ncycle \n(light-independent reactions).\n1. **Light-dependent reactions (Occurs in thylakoid membranes of chloroplasts)**\nThese reactions occur when light is absorbed by chlorophyll pigments in the thylakoid membranes.\nWater molecules are \nsplit into oxygen, protons (H+), and electrons through photolysis. The electrons move

In [86]:
milvus_similarity_rank = []
for ress in res[0]:
    milvus_similarity_rank.append(ress['distance'])

In [88]:
nolib_similarity_rank = []
for vector in content_embedding:
    similarity = cosine_similarity(vector, query_vectors[0])
    nolib_similarity_rank.append(similarity)
nolib_similarity_rank.sort(reverse=True)

Same top 10 similarity search between no lib and milvus search query

In [89]:
nolib_similarity_rank

[0.7335334926040277,
 0.6974352275914097,
 0.6621467157318642,
 0.6289659708249382,
 0.5985054868370032,
 0.44250419285031517,
 0.4384175081092816,
 0.43548533989531485,
 0.41252596684608217,
 0.3982729802545101]

In [90]:
milvus_similarity_rank

[0.7335335612297058,
 0.6974352598190308,
 0.662146806716919,
 0.6289660334587097,
 0.5985054969787598,
 0.44250422716140747,
 0.4384174644947052,
 0.4354853332042694,
 0.4125259518623352,
 0.3982730209827423]